# Supa Strikers — ball detection with SAHI (Kaggle, T4 ×2)

Runs our YOLOv8 **ball** model from the previous project with **SAHI** tiled inference (960 px slices,
0.1 overlap — the old `SahiTracker` settings) on the tactical video, both halves only. By default it
runs where the provided ball is **missing** (+/- 1 s) plus 1 in 10 frames where it is present, to
measure how far YOLO lands from it. Output: one ball per frame in pixels and pitch metres.

**Inputs:** `bar-ath-tactical` (video, ATD txt — only its ball column —, homography csv) and
`supa-strikers-models` with the ball `.pt` (its name must contain `ball`). Optionally a previous run's
output, to resume. **Settings:** GPU T4 ×2, Internet on.

**How to run:** `RUN_MODE = "smoke"` → Run All (~10 min) → check the probe crop grid (each tile should
show a ball), the distance to the provided ball and the fill rate → `RUN_MODE = "full"` → Save & Run All.
No hand labels are used anywhere (RULES.md).

In [ ]:
# ===================== CONFIG — everything you might change lives here =====================
import os

RUN_MODE = "smoke"          # "smoke": first SMOKE_FRAMES of each half (~10 min) to check everything,
                            # then "full" and run all again.
SMOKE_FRAMES = 1500

# Which frames get the ball model (SAHI costs ~6 inferences per frame)
SCOPE = "gaps"              # "gaps": where the provided ball is missing (+/- GAP_MARGIN) plus every
                            #         SAMPLE_EVERY-th other frame (to measure accuracy against it)
                            # "full": every frame of both halves
GAP_MARGIN = 25             # frames added before / after each missing-ball stretch
SAMPLE_EVERY = 10           # with SCOPE="gaps": 1 in N frames where the provided ball IS present

# Ball model + SAHI (old project's PipelineConfig: 960 px slices, 0.1 overlap, conf 0.25)
SAHI_SLICE = 960
SAHI_OVERLAP = 0.1
BALL_CONF = 0.15            # stored candidates; the pick below uses PICK_CONF
PICK_CONF = 0.25            # = old BALL_CONF_THRESHOLD
TOP_K = 3                   # candidates kept per frame (the best is not always the ball)
BALL_IMGSZ = None           # inference size per slice; None = the model's own
STANDARD_PRED = True        # also run one full-frame pass (catches a close-up ball bigger than a tile)
CHUNK = 2000                # max frames per Parquet chunk (resume granularity)

# Choosing one ball per frame
MAX_BALL_SPEED = 35.0       # m/s; a candidate further than this from the previous pick is a jump

# Outputs
MAKE_CLIPS = True
N_CLIPS, CLIP_SECONDS = 6, 8

# Match facts (DATA.md)
FPS = 25
PITCH = (105.3, 68.0)
N_VIDEO = 199319
HALVES = [(1, 11185, 90651), (2, 110808, 190197)]

DATA_DIR = os.environ.get("SS_DATA_DIR", "/kaggle/input")
WORK = os.environ.get("SS_WORK", "/kaggle/working")
BALL_MODEL_PATH = os.environ.get("SS_BALL_MODEL")   # None -> the .pt with "ball" in its name

os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
import sys
if WORK not in sys.path:
    sys.path.insert(0, WORK)


In [ ]:
import subprocess, sys

if not os.environ.get("SS_SKIP_INSTALL"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "sahi", "pyarrow"], check=False)

import glob, json, math, shutil, time, gc
import cv2
import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt

print("torch", torch.__version__, "| CUDA devices:", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
try:
    import sahi
    print("sahi", sahi.__version__)
except ImportError:
    print("sahi not installed")


## Helper modules (imported by the GPU workers)

In [ ]:
%%writefile ball_common.py
"""Shared by the notebook and the ball workers: the SAHI ball detector."""
import os

import numpy as np


def make_detector(cfg, device, tag="nb"):
    """frame (BGR) -> list of (x1, y1, x2, y2, conf). SAHI unless SS_FAKE_BALL (tests only)."""
    if os.environ.get("SS_FAKE_BALL"):
        from fake_ball import detect          # synthetic tests: reads the frame barcode
        return detect
    from sahi import AutoDetectionModel
    from sahi.predict import get_sliced_prediction
    model, err = None, None
    for mtype in ("ultralytics", "yolov8"):    # newer sahi: "ultralytics"; older: "yolov8"
        try:
            kw = dict(model_type=mtype, model_path=cfg["model_path"], confidence_threshold=cfg["conf"],
                      device=device)
            if cfg.get("imgsz"):
                kw["image_size"] = cfg["imgsz"]
            model = AutoDetectionModel.from_pretrained(**kw)
            break
        except Exception as e:                 # noqa: BLE001
            err = e
    if model is None:
        raise RuntimeError(f"could not load the ball model with SAHI: {err!r}")
    names = getattr(model, "category_names", None) or {}
    if isinstance(names, (list, tuple)):
        names = dict(enumerate(names))
    print(f"[{tag}] SAHI model loaded on {device}; classes {names}", flush=True)

    def detect(frame):
        res = get_sliced_prediction(
            frame[:, :, ::-1].copy(),          # SAHI expects RGB (the old script passed BGR)
            model, slice_height=cfg["slice"], slice_width=cfg["slice"],
            overlap_height_ratio=cfg["overlap"], overlap_width_ratio=cfg["overlap"],
            perform_standard_pred=cfg["standard_pred"], postprocess_type="NMS",
            postprocess_match_threshold=0.5, verbose=0)
        out = []
        for o in res.object_prediction_list:
            name = str(getattr(o.category, "name", "ball")).lower()
            if names and len(names) > 1 and "ball" not in name:
                continue
            out.append((o.bbox.minx, o.bbox.miny, o.bbox.maxx, o.bbox.maxy, float(o.score.value)))
        return out
    return detect


In [ ]:
%%writefile ball_worker.py
"""Ball worker: python ball_worker.py <config.json> <worker_id> <n_workers>

SAHI tiled inference with the YOLOv8 ball model (old project's SahiTracker, without ByteTrack: only
the best few boxes per frame are kept). Each worker takes the blocks whose index % n_workers ==
worker_id, decodes every frame of a block but runs the model only on the wanted frames, and writes
one Parquet per block with up to TOP_K candidates per frame. Blocks on disk are skipped (resume).
"""
import json
import os
import sys
import time

import cv2
import numpy as np
import pandas as pd
import torch

cfg = json.load(open(sys.argv[1]))
wid, nw = int(sys.argv[2]), int(sys.argv[3])
cv2.setNumThreads(2)
torch.set_num_threads(2)
device = "cuda:0" if torch.cuda.is_available() else "cpu"
want = np.load(cfg["want_path"])


from ball_common import make_detector  # noqa: E402

detect = make_detector(cfg, device, f"w{wid}")
blocks = [c for i, c in enumerate(json.load(open(cfg["chunks_path"]))) if i % nw == wid]
print(f"[w{wid}] {len(blocks)} blocks", flush=True)

for start, end in blocks:
    out_path = os.path.join(cfg["raw_dir"], f"ball_{start:06d}_{end:06d}.parquet")
    if os.path.exists(out_path):
        continue
    t0 = time.time()
    cap = cv2.VideoCapture(cfg["video"])
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    pos = int(round(cap.get(cv2.CAP_PROP_POS_FRAMES)))
    if pos != start:
        raise RuntimeError(f"seek to {start} landed on {pos}")
    rows, n_run = [], 0
    for f in range(start, end + 1):
        if not want[f]:
            if not cap.grab():
                raise RuntimeError(f"video ended at {f}")
            continue
        ok, frame = cap.read()
        if not ok:
            raise RuntimeError(f"video ended at {f}")
        n_run += 1
        dets = sorted(detect(frame), key=lambda d: -d[4])[:cfg["top_k"]]
        if not dets:
            rows.append((f, -1, np.nan, np.nan, np.nan, np.nan, np.nan))    # ran, nothing found
        for r, (x1, y1, x2, y2, c) in enumerate(dets):
            rows.append((f, r, x1, y1, x2, y2, c))
    cap.release()
    df = pd.DataFrame(rows, columns=["frame", "rank", "x1", "y1", "x2", "y2", "conf"])
    df = df.astype({"frame": np.int32, "rank": np.int8, "x1": np.float32, "y1": np.float32,
                    "x2": np.float32, "y2": np.float32, "conf": np.float32})
    tmp = out_path + ".tmp"
    df.to_parquet(tmp, index=False)
    os.replace(tmp, out_path)
    dt = time.time() - t0
    print(f"[w{wid}] block {start}-{end}: {n_run} frames run, {(df['rank'] == 0).sum()} with a ball, "
          f"{n_run / max(dt, 1e-6):.1f} fps", flush=True)
print(f"[w{wid}] done", flush=True)


## Inputs, frame ranges, resume

In [ ]:
def find_one(name, required=True):
    hits = sorted(glob.glob(os.path.join(DATA_DIR, "**", name), recursive=True))
    if not hits and required:
        raise FileNotFoundError(f"{name} not found under {DATA_DIR} — attach the dataset that holds it.")
    return hits[0] if hits else None

VIDEO = find_one("BAR-ATH_tactical.mp4")
ATD_TXT = find_one("BAR-ATH_tactical_atd.txt")            # only its BALL column is read (gaps + accuracy)
HOMO_CSV = find_one("BAR-ATH_tactical_homography.csv")

if BALL_MODEL_PATH is None:
    pts = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.pt"), recursive=True))
    ball_pts = [p for p in pts if "ball" in os.path.basename(p).lower()]
    if not ball_pts:
        raise FileNotFoundError("No ball model found: add the .pt (its name must contain 'ball') to "
                                "supa-strikers-models, or set SS_BALL_MODEL.")
    BALL_MODEL_PATH = ball_pts[0]
print("ball model:", BALL_MODEL_PATH)

RAW_DIR = os.path.join(WORK, f"ball_raw_{RUN_MODE}_{SCOPE}")
EVAL_DIR = os.path.join(WORK, "ball_eval")
CLIP_DIR = os.path.join(WORK, "ball_clips")
LOG_DIR = os.path.join(WORK, "logs")
for d in (RAW_DIR, EVAL_DIR, CLIP_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)

# Resume: chunks from an attached previous run's output are copied in and skipped.
copied = 0
for p in glob.glob(os.path.join(DATA_DIR, "**", os.path.basename(RAW_DIR), "*.parquet"), recursive=True):
    dst = os.path.join(RAW_DIR, os.path.basename(p))
    if not os.path.exists(dst):
        shutil.copy(p, dst); copied += 1

RANGES = [(h, s, e if RUN_MODE == "full" else min(e, s + SMOKE_FRAMES - 1)) for h, s, e in HALVES]
IN_PLAY = np.zeros(N_VIDEO, bool)
HALF_OF = np.zeros(N_VIDEO, np.int8)
for h, s, e in RANGES:
    IN_PLAY[s:e + 1] = True
    HALF_OF[s:e + 1] = h
print(f"mode={RUN_MODE} scope={SCOPE}  frames in play={IN_PLAY.sum():,}  resumed chunks={copied}")


## Provided ball (gaps and reference), homography, frames to run

In [ ]:
L, W = PITCH

# ---- the provided ball (normalised 0-1 -> metres), video_frame = frame_id - 1 -----------------
t0 = time.time()
ATD_BALL = np.full((N_VIDEO, 2), np.nan, np.float32)
with open(ATD_TXT) as fh:
    for line in fh:
        fid, _players, ball = line.rstrip("\n").split(":")
        vf = int(fid) - 1
        if 0 <= vf < N_VIDEO and IN_PLAY[vf] and ball[0] != "N":
            x, y = ball.split(",")[:2]
            ATD_BALL[vf] = (float(x) * L, float(y) * W)
HAS_ATD = np.isfinite(ATD_BALL[:, 0])
print(f"provided ball read in {time.time() - t0:.0f}s: present on {HAS_ATD[IN_PLAY].mean():.1%} of frames in play")

# ---- homography: normalised pitch -> pixels; inverse = pixels -> pitch ---------------------------
hdf = pd.read_csv(HOMO_CSV)
H = hdf[[f"h{r}{c}" for r in range(3) for c in range(3)]].values.astype(np.float64).reshape(-1, 3, 3)
H_VALID = hdf["valid"].values.astype(bool)
H_INV = np.full_like(H, np.nan)
H_INV[H_VALID] = np.linalg.inv(H[H_VALID])

def pix_to_pitch(frames, u, v):
    """Ball pixel -> metres, assuming the ball is ON the ground (an airborne ball lands too far)."""
    Hi = H_INV[np.asarray(frames)]
    p = np.einsum("nij,nj->ni", Hi, np.stack([u, v, np.ones_like(u)], 1).astype(np.float64))
    return (p[:, 0] / p[:, 2] * L).astype(np.float32), (p[:, 1] / p[:, 2] * W).astype(np.float32)

def pitch_to_pix(frame, xm, ym):
    p = H[frame] @ np.array([xm / L, ym / W, 1.0])
    return p[0] / p[2], p[1] / p[2]

# ---- frames to process ------------------------------------------------------------------------
want = np.zeros(N_VIDEO, bool)
if SCOPE == "full":
    want[IN_PLAY] = True
else:
    miss = IN_PLAY & ~HAS_ATD
    d = np.diff(np.r_[0, miss.astype(np.int8), 0])
    for a, b in zip(np.flatnonzero(d == 1), np.flatnonzero(d == -1)):        # [a, b) missing
        want[max(a - GAP_MARGIN, 0):min(b + GAP_MARGIN, N_VIDEO)] = True
    sample = IN_PLAY & HAS_ATD & (np.arange(N_VIDEO) % SAMPLE_EVERY == 0)
    want |= sample
want &= IN_PLAY

# Blocks of CHUNK consecutive frames. A worker decodes every frame of a block (cheap) but runs the
# ball model only on the wanted ones, so sampled single frames cost no seek.
CHUNKS = []
for h, s_, e_ in RANGES:
    for a_ in range(s_, e_ + 1, CHUNK):
        b_ = min(a_ + CHUNK - 1, e_)
        if want[a_:b_ + 1].any():
            CHUNKS.append([int(a_), int(b_)])
WANT = want
np.save(os.path.join(WORK, "ball_want.npy"), WANT)
print(f"frames to run: {want.sum():,} ({want.sum() / max(IN_PLAY.sum(), 1):.0%} of play) in {len(CHUNKS)} blocks; "
      f"{(want & ~HAS_ATD).sum():,} without the provided ball, {(want & HAS_ATD).sum():,} with it (accuracy check)")


## Video sanity check

In [ ]:
cap = cv2.VideoCapture(VIDEO)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
VID_W, VID_H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"video: {n_frames:,} frames, {cap.get(cv2.CAP_PROP_FPS):.2f} fps, {VID_W}x{VID_H}")
if abs(n_frames - N_VIDEO) > 2:
    print("WARNING: frame count differs from DATA.md — check you attached the TACTICAL video.")
bad = []
for f in (RANGES[0][1], (RANGES[0][1] + RANGES[0][2]) // 2 + 7, (RANGES[-1][1] + RANGES[-1][2]) // 2 + 13):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f); ok, direct = cap.read()
    cap.set(cv2.CAP_PROP_POS_FRAMES, f - 60)
    for _ in range(60):
        cap.grab()
    ok2, seq = cap.read()
    if not (ok and ok2):
        raise RuntimeError(f"could not read frame {f}")
    diff = float(np.abs(direct.astype(np.int16) - seq.astype(np.int16)).mean())
    print(f"  frame {f}: direct-seek vs sequential mean abs diff = {diff:.2f}")
    if diff > 1.0:
        bad.append(f)
cap.release()
if bad:
    raise RuntimeError(f"Seeking is not frame-accurate at {bad}.")
print("seek check passed")


## Probe — 24 frames: does the model see the ball, how fast is it?
Check the crop grid: every tile should show a ball. Note the projected time for the full pass.

In [ ]:
from ball_common import make_detector

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
cfg_w = dict(video=VIDEO, model_path=os.path.abspath(BALL_MODEL_PATH), conf=BALL_CONF, imgsz=BALL_IMGSZ,
             slice=SAHI_SLICE, overlap=SAHI_OVERLAP, standard_pred=STANDARD_PRED, top_k=TOP_K,
             want_path=os.path.join(WORK, "ball_want.npy"), raw_dir=RAW_DIR,
             chunks_path=os.path.join(WORK, "ball_chunks.json"))
detect = make_detector(cfg_w, DEVICE, "probe")

# 24 frames where the provided ball IS present: speed, and a first look at agreement
cand = np.flatnonzero(WANT & HAS_ATD)
probe = cand[np.linspace(0, len(cand) - 1, min(24, len(cand))).astype(int)] if len(cand) else []
cap = cv2.VideoCapture(VIDEO)
tiles, dists, t0, n_found = [], [], time.time(), 0
for f in probe:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(f)); ok, fr = cap.read()
    dets = sorted(detect(fr), key=lambda d: -d[4])
    if dets and dets[0][4] >= PICK_CONF:
        n_found += 1
        x1, y1, x2, y2, c = dets[0]
        u, v = (x1 + x2) / 2, (y1 + y2) / 2
        xm, ym = pix_to_pitch([f], np.array([u]), np.array([v]))
        dists.append(float(np.hypot(xm[0] - ATD_BALL[f, 0], ym[0] - ATD_BALL[f, 1])))
        cu, cv_ = int(u), int(v)
        crop = fr[max(cv_ - 40, 0):cv_ + 40, max(cu - 40, 0):cu + 40].copy()
        if crop.size:
            tiles.append(cv2.resize(crop, (96, 96)))
probe_fps = len(probe) / max(time.time() - t0, 1e-6)
cap.release()
print(f"probe: {n_found}/{len(probe)} frames with a ball (conf >= {PICK_CONF}); "
      f"{probe_fps:.1f} frames/s on one {'GPU' if DEVICE != 'cpu' else 'CPU'} (incl. seeks)")
if dists:
    print(f"distance to the provided ball: median {np.median(dists):.2f} m, within 2 m {np.mean(np.array(dists) <= 2):.0%}")
if tiles:
    tiles += [np.full((96, 96, 3), 255, np.uint8)] * (-len(tiles) % 8)
    grid = np.vstack([np.hstack(tiles[r:r + 8]) for r in range(0, len(tiles), 8)])
    plt.figure(figsize=(8, grid.shape[0] / grid.shape[1] * 8 + 0.3))
    plt.imshow(cv2.cvtColor(grid, cv2.COLOR_BGR2RGB)); plt.axis("off")
    plt.title("best ball candidate per probe frame (80 px around it) — each should show a ball", loc="left", fontsize=9)
    plt.savefig(os.path.join(EVAL_DIR, "probe_crops.png"), dpi=110, bbox_inches="tight"); plt.show()
n_workers = max(1, torch.cuda.device_count())
print(f"Projected ball pass: {WANT.sum():,} frames on {n_workers} worker(s) ≈ "
      f"{WANT.sum() / (max(probe_fps, 0.1) * n_workers) / 3600:.1f} h (+ decoding)")
del detect
gc.collect(); torch.cuda.empty_cache()


## Ball pass on both GPUs (resumable)

In [ ]:
import subprocess

def chunk_path(s, e):
    return os.path.join(RAW_DIR, f"ball_{s:06d}_{e:06d}.parquet")

todo = [c for c in CHUNKS if not os.path.exists(chunk_path(*c))]
print(f"{len(CHUNKS) - len(todo)}/{len(CHUNKS)} blocks already on disk, {len(todo)} to run")
if todo:
    save = lambda o, p: json.dump(o, open(p, "w"), indent=1)
    save(todo, cfg_w["chunks_path"])
    save(cfg_w, os.path.join(WORK, "ball_worker_cfg.json"))
    n_workers = max(1, torch.cuda.device_count())
    procs, logs = [], []
    for w in range(n_workers):
        env = dict(os.environ)
        if torch.cuda.is_available():
            env["CUDA_VISIBLE_DEVICES"] = str(w)
        logs.append(os.path.join(LOG_DIR, f"ball_worker{w}.log"))
        procs.append(subprocess.Popen([sys.executable, os.path.join(WORK, "ball_worker.py"),
                                       os.path.join(WORK, "ball_worker_cfg.json"), str(w), str(n_workers)],
                                      env=env, cwd=WORK, stdout=open(logs[-1], "w"), stderr=subprocess.STDOUT))
    t0 = time.time()
    while any(p.poll() is None for p in procs):
        time.sleep(60 if RUN_MODE == "full" else 15)
        done = sum(os.path.exists(chunk_path(*c)) for c in CHUNKS)
        el = (time.time() - t0) / 60
        fin = len(todo) - (len(CHUNKS) - done)
        eta = f"ETA ≈ {el / fin * (len(CHUNKS) - done):.0f} min" if fin else "loading models / first block"
        print(f"[{el:6.1f} min] blocks {done}/{len(CHUNKS)}  {eta}", flush=True)
    for w, p in enumerate(procs):
        if p.returncode != 0:
            print(open(logs[w]).read()[-3000:])
            raise RuntimeError(f"ball worker {w} failed (exit {p.returncode}); log above. Rerun this cell to resume.")
    print(f"ball pass finished in {(time.time() - t0) / 60:.1f} min")
missing = [c for c in CHUNKS if not os.path.exists(chunk_path(*c))]
assert not missing, f"missing blocks: {missing[:5]}"


## One ball per frame

In [ ]:
C = pd.concat([pd.read_parquet(chunk_path(*c)) for c in CHUNKS], ignore_index=True)
RAN = np.zeros(N_VIDEO, bool)
RAN[C.frame.values] = True
C = C[(C["rank"] >= 0) & (C.conf >= PICK_CONF)].copy()
C["u"] = (C.x1 + C.x2) / 2
C["v"] = (C.y1 + C.y2) / 2
C["x_m"], C["y_m"] = pix_to_pitch(C.frame.values, C.u.values.astype(np.float64), C.v.values.astype(np.float64))
C = C.sort_values(["frame", "conf"], ascending=[True, False]).reset_index(drop=True)
print(f"frames run: {RAN.sum():,}; candidates with conf >= {PICK_CONF}: {len(C):,} on {C.frame.nunique():,} frames")

# One ball per frame: the most confident candidate, unless it implies a jump faster than
# MAX_BALL_SPEED from the last pick (within 1 s) and another candidate does not.
picked, last_f, last_xy, n_switched = [], None, None, 0
fr_all = C.frame.to_numpy()
bounds = np.r_[0, np.flatnonzero(np.diff(fr_all)) + 1, len(C)]
for a, b in zip(bounds[:-1], bounds[1:]):
    g = C.iloc[a:b]
    f = int(g.frame.iloc[0])
    choice, jump = 0, False
    if last_f is not None and f - last_f <= FPS:
        dt = (f - last_f) / FPS
        sp = np.hypot(g.x_m.values - last_xy[0], g.y_m.values - last_xy[1]) / dt
        ok = np.isfinite(sp) & (sp <= MAX_BALL_SPEED)
        if not ok[0]:
            if ok.any():
                choice = int(np.flatnonzero(ok)[0]); n_switched += 1
            else:
                jump = True
    r = g.iloc[choice]
    picked.append((f, r.u, r.v, r.conf, r.x_m, r.y_m, b - a, jump))
    if not jump:
        last_f, last_xy = f, (r.x_m, r.y_m)
B = pd.DataFrame(picked, columns=["frame", "u", "v", "conf", "x_m", "y_m", "n_cand", "jump"])
B["half"] = HALF_OF[B.frame.values]
B["has_atd"] = HAS_ATD[B.frame.values]
B["atd_x"], B["atd_y"] = ATD_BALL[B.frame.values, 0], ATD_BALL[B.frame.values, 1]
B["dist_atd"] = np.hypot(B.x_m - B.atd_x, B.y_m - B.atd_y)
print(f"picked a ball on {len(B):,} frames; {n_switched} picks moved to a plausible 2nd/3rd candidate; "
      f"{int(B.jump.sum())} flagged as jumps (no plausible candidate)")


## How good is it? (against the provided ball where both exist; fill rate where it is missing)
Positions use the homography and assume the ball is on the ground: a ball in the air lands too far.

In [ ]:
YOLO = np.zeros(N_VIDEO, bool); YOLO[B.frame.values] = True
DIST = np.full(N_VIDEO, np.nan); DIST[B.frame.values] = B.dist_atd.values
JUMP = np.zeros(N_VIDEO, bool); JUMP[B.frame.values[B.jump.values]] = True

both = RAN & HAS_ATD
gapf = RAN & ~HAS_ATD & IN_PLAY
d = DIST[both & YOLO]
res = {"run_mode": RUN_MODE, "scope": SCOPE, "frames_run": int(RAN.sum()),
       "ball_found_share": float(YOLO[RAN].mean()) if RAN.any() else float("nan"),
       "on_provided_frames": int(both.sum()),
       "found_where_provided": float(YOLO[both].mean()) if both.any() else float("nan"),
       "median_dist_to_provided_m": float(np.median(d)) if len(d) else float("nan"),
       "within_1m": float(np.mean(d <= 1)) if len(d) else float("nan"),
       "within_2m": float(np.mean(d <= 2)) if len(d) else float("nan"),
       "within_5m": float(np.mean(d <= 5)) if len(d) else float("nan"),
       "on_missing_frames": int(gapf.sum()),
       "filled_where_missing": float(YOLO[gapf].mean()) if gapf.any() else float("nan"),
       "jump_share": float(JUMP[YOLO].mean()) if YOLO.any() else float("nan")}

# Fill rate by length of the provided ball's gap
miss = IN_PLAY & ~HAS_ATD
dd = np.diff(np.r_[0, miss.astype(np.int8), 0])
rows = []
for a, b in zip(np.flatnonzero(dd == 1), np.flatnonzero(dd == -1)):
    n = b - a
    ran_n = int(RAN[a:b].sum())
    if ran_n:
        rows.append((a, n, ran_n, int(YOLO[a:b].sum())))
G = pd.DataFrame(rows, columns=["start", "length", "frames_run", "frames_filled"])
G["bucket"] = pd.cut(G.length, [0, 5, 25, 125, 10 ** 7], labels=["1-5", "6-25", "26-125", ">125"])
by = G.groupby("bucket", observed=False)[["frames_run", "frames_filled"]].sum()
by["fill_share"] = np.where(by.frames_run > 0, by.frames_filled / by.frames_run.clip(lower=1), np.nan)
res["fill_by_gap_length"] = {str(k): (None if np.isnan(v) else round(float(v), 3)) for k, v in by.fill_share.items()}
G.to_csv(os.path.join(EVAL_DIR, f"gaps_{RUN_MODE}.csv"), index=False)
json.dump(res, open(os.path.join(EVAL_DIR, f"summary_{RUN_MODE}.json"), "w"), indent=2)
for k, v in res.items():
    print(f"  {k:<28} {v:.3f}" if isinstance(v, float) else f"  {k:<28} {v}")


## Plots

In [ ]:
INK, MUTED, GRID, C1 = "#0b0b0b", "#52514e", "#e4e3df", "#2a78d6"
plt.rcParams.update({"axes.edgecolor": GRID, "axes.labelcolor": MUTED, "xtick.color": MUTED,
                     "ytick.color": INK, "font.size": 9, "axes.titlelocation": "left"})
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
ax = axes[0]
dd_ = DIST[both & YOLO]
if len(dd_):
    ax.hist(np.clip(dd_, 0, 10), bins=40, color=C1)
    ax.axvline(np.median(dd_), color=INK, lw=1)
    ax.text(np.median(dd_), ax.get_ylim()[1] * 0.92, f"  median {np.median(dd_):.2f} m", fontsize=8)
ax.set_title("Distance YOLO ball - provided ball (frames with both; clipped at 10 m)")
ax.set_xlabel("metres"); ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True)
ax = axes[1]
ax.bar(by.index.astype(str), np.nan_to_num(by.fill_share.values), color=C1)
for i, (v, n) in enumerate(zip(by.fill_share.values, by.frames_run.values)):
    ax.text(i, 0 if np.isnan(v) else v, "no gaps" if np.isnan(v) else f"{v:.0%}\n({n:,} fr)",
            ha="center", va="bottom", fontsize=8)
ax.set_ylim(0, 1.15); ax.set_title("Share of missing-ball frames where YOLO finds a ball, by gap length")
ax.set_xlabel("gap length (frames)"); ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True)
for a_ in axes:
    for s_ in ("top", "right"):
        a_.spines[s_].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(EVAL_DIR, f"ball_eval_{RUN_MODE}.png"), dpi=120, bbox_inches="tight"); plt.show()


## Clips of the longest gaps YOLO fills (white = provided ball, green = YOLO)

In [ ]:
if MAKE_CLIPS and len(G):
    n_clip = CLIP_SECONDS * FPS
    picks = G[G.frames_filled > 0].sort_values("length", ascending=False).head(N_CLIPS)
    bf = dict(zip(B.frame.values, zip(B.u.values, B.v.values, B.conf.values)))
    for r in picks.itertuples():
        f0 = int(max(r.start - FPS, 0)); f1 = int(min(f0 + n_clip, N_VIDEO))
        path = os.path.join(CLIP_DIR, f"gap_{r.start}_{r.length}.mp4")
        cap = cv2.VideoCapture(VIDEO); cap.set(cv2.CAP_PROP_POS_FRAMES, f0)
        out = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (VID_W // 2, VID_H // 2))
        for f in range(f0, f1):
            ok, fr = cap.read()
            if not ok:
                break
            if HAS_ATD[f] and H_VALID[f]:
                u, v = pitch_to_pix(f, *ATD_BALL[f])
                if np.isfinite(u):
                    cv2.circle(fr, (int(u), int(v)), 16, (255, 255, 255), 3)          # provided: white
            if f in bf:
                u, v, c = bf[f]
                cv2.circle(fr, (int(u), int(v)), 11, (60, 220, 60), 3)                # YOLO: green
                cv2.putText(fr, f"{c:.2f}", (int(u) + 14, int(v) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (60, 220, 60), 2)
            label = f"frame {f}  provided ball: {'yes' if HAS_ATD[f] else 'MISSING'}  YOLO: {'yes' if f in bf else 'no'}"
            cv2.putText(fr, label, (20, VID_H - 30), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 6)
            cv2.putText(fr, label, (20, VID_H - 30), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
            out.write(cv2.resize(fr, (VID_W // 2, VID_H // 2)))
        out.release(); cap.release()
        print("clip:", path)


## Export

In [ ]:
out = os.path.join(WORK, f"ball_yolo_{RUN_MODE}.parquet")
B[["frame", "half", "u", "v", "conf", "x_m", "y_m", "n_cand", "jump"]].to_parquet(out, index=False)
C[["frame", "rank", "x1", "y1", "x2", "y2", "conf", "u", "v", "x_m", "y_m"]].to_parquet(
    os.path.join(WORK, f"ball_candidates_{RUN_MODE}.parquet"), index=False)
np.save(os.path.join(WORK, f"ball_frames_run_{RUN_MODE}.npy"), RAN)
print("wrote", out, f"({len(B):,} frames with a ball)")
print("     ", os.path.join(WORK, f"ball_candidates_{RUN_MODE}.parquet"))
print("      ball_frames_run_*.npy marks every frame the model ran on (no ball there = really not found)")
for p in sorted(glob.glob(os.path.join(WORK, "ball_eval", "*")) + glob.glob(os.path.join(CLIP_DIR, "*"))):
    print(f"  {os.path.relpath(p, WORK):50s} {os.path.getsize(p) / 1e6:7.1f} MB")
